**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->
# Computer Vision — Lezione 15
## Object detection

**Lessico del problema.** Un’istanza è un singolo oggetto; la sua bounding box è un rettangolo. GT significa ground truth, annotazione di riferimento. Lo score `s` ordina le predizioni: il suo significato dipende dalla testa del detector.

09 novembre 2026. **Notebook dimostrativo completo:** tutte le 13 celle di codice sono già scritte. Gli esempi numerici sono assegnati, non output di un detector addestrato. Gli identificatori Cxx contano tutte le 27 celle, compreso questo titolo.

N01–N12 sono letture guidate con spiegazioni brevi già disponibili; F01–F04 sono una sintesi già svolta. Si può seguire la demo e rieseguirla per il ripasso. 

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

<!-- C02 -->
## Colab o Jupyter
Caricare questo singolo file in Google Colab, scegliere runtime Python CPU e partire da C03. Non servono file esterni, pesi, GPU, installazioni o Drive. In Jupyter servono NumPy e IPython. Eseguire in ordine per la prima prova; i blocchi indicano i prerequisiti. Si usano box continui `xyxy`, senza `+1` nell’area. Tutti i dati sono sintetici e pubblici. Le letture guidate seguenti contengono spiegazioni disponibili agli studenti. Non occorre compilare campi o preparare una relazione per questa lezione.

In [ ]:
# C03
import sys, json, time
from pathlib import Path
import numpy as np
from IPython.display import display, Markdown
np.set_printoptions(precision=6, suppress=True)
START = time.perf_counter()
RESULTS = {"python": sys.version.split()[0], "numpy": np.__version__, "dataset": "box sintetici assegnati; nessun detector"}
def show(name, value):
    print(name, value)
print("Ambiente pronto — CPU; nessun download", RESULTS)


<!-- C04 -->
## N01 — Coordinate e trasformazioni
C05 richiede C03.

**Lettura guidata.** Il box xyxy `(4, 6, 12, 10)` ha centro `(8, 8)`, larghezza 8 e altezza 4. Con scale `(2, 0.5)` e padding `(3, 7)`, diventa `(11, 10, 27, 12)`. Sottrarre il padding e dividere per le scale restituisce il box iniziale, con errore massimo 0 in C05. Immagine e annotazioni devono subire la stessa trasformazione; un eventuale clipping perde informazione e può impedire l’inversione completa.

In [ ]:
# C05
def xyxy_to_cxcywh(box):
    b=np.asarray(box, dtype=float)
    return np.r_[(b[:2]+b[2:])/2, b[2:]-b[:2]]
b=np.array([4.,6.,12.,10.]); scale=np.array([2.,.5,2.,.5]); pad=np.array([3.,7.,3.,7.])
transformed=b*scale+pad; recovered=(transformed-pad)/scale
show("centro,w,h:",xyxy_to_cxcywh(b)); show("box trasformato:",transformed); show("inversa:",recovered)
assert np.allclose(recovered,b)
RESULTS["coordinate"]={"center":xyxy_to_cxcywh(b).tolist(),"transformed":transformed.tolist(),"inverse_error":float(np.max(abs(recovered-b)))}


<!-- C06 -->
## N02 — IoU e GIoU

**Prima dei conti.** IoU significa Intersection over Union: area comune divisa per area dell’unione. GIoU, Generalized IoU, aggiunge una penalità basata sul rettangolo che contiene i due box. Si usano box continue, senza `+1` nell’area.

C07 richiede C03.

**Lettura guidata.** I box A e B hanno area 16, intersezione 8 e unione 24: `IoU=1/3`. A e D sono disgiunti: intersezione 0, unione 32 e rettangolo contenitore di area 40; perciò `IoU=0` e `GIoU=−0.2`. C07 usa coordinate continue e rifiuta box invertiti o degeneri, senza aggiungere 1 alle dimensioni.

In [ ]:
# C07
def checked(box):
    b=np.asarray(box,dtype=float)
    if b.shape!=(4,) or not np.isfinite(b).all() or not np.all(b[2:]>b[:2]):
        raise ValueError("Serve box xyxy finito con larghezza e altezza positive")
    return b
def iou(a,b):
    a,b=checked(a),checked(b)
    inter=np.prod(np.maximum(0,np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2])))
    union=np.prod(a[2:]-a[:2])+np.prod(b[2:]-b[:2])-inter
    return float(inter/union)
def giou(a,b):
    a,b=checked(a),checked(b)
    inter=np.prod(np.maximum(0,np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2])))
    union=np.prod(a[2:]-a[:2])+np.prod(b[2:]-b[:2])-inter
    enclosing=np.prod(np.maximum(a[2:],b[2:])-np.minimum(a[:2],b[:2]))
    return float(inter/union-(enclosing-union)/enclosing)
A=[0,0,4,4]; B=[2,0,6,4]; D=[6,0,10,4]
RESULTS["overlap"]={"iou_AB":iou(A,B),"iou_AD":iou(A,D),"giou_AD":giou(A,D)}
print(RESULTS["overlap"])
assert np.isclose(iou(A,B),1/3) and np.isclose(giou(A,D),-.2)


<!-- C08 -->
## N03 — Encode e decode

**Prima della codifica.** Anchor significa box di riferimento prefissata; offset è una correzione relativa. Encode trasforma la box in offset; decode applica l’inversa. Il target è la box annotata desiderata, distinta dall’anchor e dalla predizione.

C09 richiede C03/C05.

**Lettura guidata.** Gli offset del caso assegnato sono `(0.5, −0.25, ln(2), −ln(2))`, circa `(0.5, −0.25, 0.693147, −0.693147)`. I primi due spostamenti sono normalizzati per le dimensioni dell’anchor; gli ultimi due sono logaritmi dei rapporti di larghezza e altezza. La decodifica di C09 ricostruisce il target xyxy `(8, 6, 16, 10)`. I pesi aggiuntivi usati da alcuni coder reali sono qui tutti 1.

In [ ]:
# C09
def encode(anchor, box):
    a=xyxy_to_cxcywh(anchor); b=xyxy_to_cxcywh(box)
    return np.r_[(b[:2]-a[:2])/a[2:],np.log(b[2:]/a[2:])]
def decode(anchor, delta):
    a=xyxy_to_cxcywh(anchor); d=np.asarray(delta)
    c=a[:2]+d[:2]*a[2:]; wh=a[2:]*np.exp(d[2:])
    return np.r_[c-wh/2,c+wh/2]
anchor=np.array([8,6,12,14]); target=np.array([8,6,16,10])
t=encode(anchor,target); rec=decode(anchor,t)
print("offset",t,"box ricostruito",rec)
assert np.allclose(rec,target)
RESULTS["anchor"]={"delta":t.tolist(),"decoded":rec.tolist()}


<!-- C10 -->
## N04 — Assignment durante il training

**Prima dell’assignment.** Assignment è l’assegnazione dei target ai candidati: oggetto GT, sfondo oppure ignore. Ignore esclude il candidato dai termini previsti dalla regola; più anchor positive possono apprendere lo stesso oggetto.

C11 richiede C03/C07.

**Lettura guidata.** Le quattro IoU sono `(1, 9/11, 3/7, 0)`, circa `(1, 0.818182, 0.428571, 0)`. La regola dichiarata in C11 assegna le etichette `(1, 1, −1, 0)`: due positivi, un ignore e un negativo. Durante il training più anchor possono usare lo stesso GT; questo non è il matching che conta le TP in valutazione.

In [ ]:
# C11
gt=np.array([0,0,10,10]); anchors=np.array([[0,0,10,10],[1,0,11,10],[4,0,14,10],[20,0,30,10]])
overlaps=np.array([iou(a,gt) for a in anchors]); positive=overlaps>=.7
positive[overlaps==overlaps.max()]=True
labels=np.where(positive,1,np.where(overlaps<.3,0,-1))
print("IoU",overlaps,"label (+1 positivo,0 negativo,-1 ignore)",labels)
assert labels.tolist()==[1,1,-1,0]
RESULTS["assignment"]={"ious":overlaps.tolist(),"labels":labels.tolist()}


<!-- C12 -->
## N05 — NMS greedy

**Prima della NMS.** Non-Maximum Suppression è un filtro di duplicati fra predizioni. Greedy significa scegliere prima lo score maggiore, poi sopprimere candidati sovrapposti secondo la regola. Non consulta i GT.

C13 richiede C03/C07.

**Lettura guidata.** I primi due box hanno `IoU=9/11≈0.818182`, maggiore della soglia NMS `0.5`. Il box con score 0.9 viene mantenuto, quello con score 0.8 viene soppresso e il terzo, disgiunto, rimane: C13 restituisce gli indici `[0, 2]`. La NMS confronta predizioni e score, senza consultare i GT.

In [ ]:
# C13
def nms(boxes,scores,threshold=.5,classes=None):
    boxes=np.asarray(boxes,float); scores=np.asarray(scores,float)
    if classes is None: classes=np.zeros(len(boxes),dtype=int)
    order=sorted(range(len(boxes)),key=lambda i:(-scores[i],i)); keep=[]
    while order:
        i=order.pop(0); keep.append(i)
        order=[j for j in order if classes[j]!=classes[i] or iou(boxes[i],boxes[j])<=threshold]
    return keep
boxes=np.array([[0,0,10,10],[1,0,11,10],[20,0,30,10]])
scores=np.array([.9,.8,.7]); keep=nms(boxes,scores)
print("IoU primi due",iou(boxes[0],boxes[1]),"keep",keep)
assert keep==[0,2]
RESULTS["nms"]={"keep":keep,"iou01":iou(boxes[0],boxes[1])}


<!-- C14 -->
## N06 — Classi e parità

**Due politiche.** Class-aware confronta box della stessa classe; class-agnostic ignora la classe. Pareggio significa score uguale: occorre una regola deterministica, qui l’indice originale.

C15 richiede C13.

**Lettura guidata.** Con classi `(1, 2, 1)`, la NMS class-aware conserva `[0, 1, 2]`; quella class-agnostic conserva `[0, 2]`. Pareggiando gli score dei primi due box, questa implementazione sceglie l’indice iniziale minore e conserva `[0, 2]`. Nel caso al confine `IoU=threshold=1/3` restano entrambi i box, perché la soppressione usa `>` e non `>=`. Regola dei pareggi e confronto al confine vanno verificati nell’implementazione effettiva.

In [ ]:
# C15
classes=np.array([1,2,1]); aware=nms(boxes,scores,classes=classes); agnostic=nms(boxes,scores)
tied=nms(boxes,[.9,.9,.7]); equal_boundary=nms([[0,0,4,4],[2,0,6,4]],[.9,.8],threshold=1/3)
print("class-aware",aware,"agnostic",agnostic,"parità",tied,"IoU==soglia",equal_boundary)
assert aware==[0,1,2] and tied==[0,2] and equal_boundary==[0,1]
RESULTS["nms"].update(aware=aware,agnostic=agnostic,tied=tied,boundary=equal_boundary)


<!-- C16 -->
## N07 — Matching valutativo

**Prima del matching.** Il matching valutativo associa predizioni e GT della stessa immagine e classe. TP è una predizione abbinata validamente, FP una predizione non abbinata, FN un GT rimasto libero. Il ranking è l’ordinamento per score.

C17 richiede C03/C07.

**Lettura guidata.** A soglia valutativa IoU 0.5, il ranking p1–p5 produce la sequenza `TP, FP, TP, TP, FP`. p2 duplica un GT già associato a p1; p5 non sovrappone il GT di B. Il ranking è globale per la classe, ma l’associazione resta nella stessa immagine. C17 consuma ciascun GT ordinario al massimo una volta e non gestisce crowd o ignore.

In [ ]:
# C17
GT=[dict(image="A",cls=1,box=[0,0,10,10]),dict(image="B",cls=1,box=[20,20,30,30]),dict(image="C",cls=1,box=[40,40,50,50])]
PRED=[dict(id="p1",image="A",cls=1,box=[0,0,10,10],score=.95),dict(id="p2",image="A",cls=1,box=[1,0,11,10],score=.90),dict(id="p3",image="B",cls=1,box=[21,20,31,30],score=.80),dict(id="p4",image="C",cls=1,box=[40,40,48,50],score=.70),dict(id="p5",image="B",cls=1,box=[0,0,5,5],score=.60)]
def match_eval(preds,gt,threshold=.5,cls=1):
    gg=[g for g in gt if g['cls']==cls]; used=set(); tp=[]; rows=[]
    pp=sorted([(i,p) for i,p in enumerate(preds) if p['cls']==cls],key=lambda ip:(-ip[1]['score'],ip[0]))
    for _,p in pp:
        candidates=[(iou(p['box'],g['box']),j) for j,g in enumerate(gg) if g['image']==p['image'] and j not in used]
        best=max(candidates,key=lambda q:(q[0],-q[1])) if candidates else (0.,-1)
        ok=best[1]>=0 and best[0]>=threshold
        if ok: used.add(best[1])
        tp.append(int(ok)); rows.append((p['id'],p['score'],int(ok),best[0]))
    return np.asarray(tp,dtype=int),len(gg),rows
flags,ngt,rows=match_eval(PRED,GT)
for row in rows:print(row)
assert flags.tolist()==[1,0,1,1,0]
RESULTS['matching']={'tp':flags.tolist(),'gt':ngt,'rows':rows}


<!-- C18 -->
## N08 — Precision/recall e AP integrale

**Prima di AP.** Precision è la frazione corretta delle predizioni mantenute; recall è la frazione di GT recuperata. L’inviluppo prende per ogni recall la massima precisione fra i punti che raggiungono almeno quel recall. Average Precision (AP) riassume il ranking con la convenzione dichiarata; non è la media aritmetica delle precision.

C19 richiede C17.

**Lettura guidata.** Prima della NMS i TP cumulati sono `(1, 1, 2, 3, 3)` e gli FP cumulati `(0, 1, 1, 1, 2)`. La precision è `(1, 1/2, 2/3, 3/4, 3/5)` e la recall `(1/3, 1/3, 2/3, 1, 1)`. L’inviluppo assegna precision 1 al primo incremento di recall e 3/4 ai successivi: `AP integrale=1/3+(2/3)*(3/4)=5/6`. C19 non usa una regola dei trapezi.

In [ ]:
# C19
def pr_ap(tp,n_gt):
    tp=np.asarray(tp,int); cum=np.cumsum(tp); precision=cum/np.arange(1,len(tp)+1)
    recall=cum/n_gt if n_gt else np.full(len(tp),np.nan)
    if n_gt==0:return precision,recall,float('nan')
    mr=np.r_[0.,recall,1.]; mp=np.r_[0.,precision,0.]
    mp=np.maximum.accumulate(mp[::-1])[::-1]; idx=np.where(mr[1:]!=mr[:-1])[0]
    ap=float(np.sum((mr[idx+1]-mr[idx])*mp[idx+1]))
    return precision,recall,ap
precision,recall,ap=pr_ap(flags,ngt)
print('P',precision,'R',recall,'AP integrale',ap)
assert np.isclose(ap,5/6)
RESULTS['pr']={'precision':precision.tolist(),'recall':recall.tolist(),'ap_integrale':ap}


<!-- C20 -->
## N09 — AP a 101 livelli di recall
C21 richiede C19.

**Lettura guidata.** Nel caso assegnato, 34 dei 101 livelli di recall usano precision 1 e gli altri 67 usano 0.75: `AP101=(34+67*0.75)/101=337/404≈0.834158416`. È una convenzione diversa dall’AP integrale `5/6`. Lo sweep IoU di C21 resta semplificato: non include gestione COCO di crowd, ignore, fasce d’area e maxDet, quindi non è COCOeval completo.

In [ ]:
# C21
def ap101(tp,n_gt):
    p,r,_=pr_ap(tp,n_gt)
    if n_gt==0:return float('nan')
    return float(np.mean([np.max(p[r>=q]) if np.any(r>=q) else 0. for q in np.linspace(0,1,101)]))
ap_sampled=ap101(flags,ngt); sweep=[]
for threshold in np.round(np.arange(.5,.951,.05),2):
    ft,nt,_=match_eval(PRED,GT,threshold)
    sweep.append({'iou':float(threshold),'ap101':ap101(ft,nt)})
print('AP101',ap_sampled,'integrale',ap);print(sweep)
assert np.isclose(ap_sampled,(34+67*.75)/101)
RESULTS['ap101']={'before':ap_sampled,'sweep_simplified':sweep,'mean_simplified':float(np.mean([s['ap101'] for s in sweep]))}


<!-- C22 -->
## N10 — NMS e AP
C23 richiede C13/C17/C19/C21.

**Lettura guidata.** La NMS per immagine rimuove p2: restano p1, p3, p4 e p5. A IoU valutativa 0.5, i primi tre sono TP e p5 è ancora FP; AP integrale e AP101 valgono entrambe 1, perché p5 arriva dopo aver raggiunto recall 1. Cambiando separatamente la soglia valutativa a 0.85, solo p1 è TP e `AP101=34/101≈0.336633663`. C23 distingue la soglia NMS dalla soglia che definisce la metrica.

In [ ]:
# C23
def nms_by_image(preds,threshold=.5):
    kept=[]
    for image in sorted(set(p['image'] for p in preds)):
        pp=[p for p in preds if p['image']==image]
        inds=nms([p['box'] for p in pp],[p['score'] for p in pp],threshold,[p['cls'] for p in pp])
        kept += [pp[i] for i in inds]
    return sorted(kept,key=lambda p:-p['score'])
filtered=nms_by_image(PRED); fl,n,_=match_eval(filtered,GT)
pp,rr,aa=pr_ap(fl,n); strict,_,strict_rows=match_eval(filtered,GT,.85)
print('keep',[p['id'] for p in filtered], 'TP',fl,'AP integrale',aa,'AP101',ap101(fl,n))
print('IoUeval=.85',strict,'AP101',ap101(strict,n))
assert np.isclose(aa,1) and strict.tolist()==[1,0,0,0]
RESULTS['after_nms']={'keep':[p['id'] for p in filtered],'tp':fl.tolist(),'ap_integrale':aa,'ap101':ap101(fl,n),'strict_tp':strict.tolist(),'strict_ap101':ap101(strict,n)}


<!-- C24 -->
## N11 — Casi limite
C25 richiede C07/C17/C19/C21.

**Lettura guidata.** Con GT presenti e nessuna predizione, AP vale 0. Una classe senza GT validi ha invece AP non applicabile: C25 la riporta come `N/A`, non come zero. Un box invertito genera `ValueError`. Una soglia NMS diversa non garantisce AP migliore o insiemi mantenuti annidati: il procedimento greedy cambia anche i confronti successivi.

In [ ]:
# C25
empty,nn,_=match_eval([],GT); no_gt,zero,_=match_eval(PRED,GT,cls=2)
assert ap101(empty,nn)==0 and np.isnan(ap101(no_gt,zero))
try:
    iou([4,0,2,4],A)
except ValueError:
    print('Box invertito: rifiutato correttamente')
else:raise AssertionError('Un box invalido è stato accettato')
print('nessuna predizione, GT3:',ap101(empty,nn),'classe senza GT:',ap101(no_gt,zero))
RESULTS['edge_cases']={'no_predictions_ap':ap101(empty,nn),'no_gt':'N/A','invalid_box':'rejected'}


<!-- C26 -->
## N12 — Interpretazione e sintesi svolta
**Lettura guidata.** L’assignment usa i GT per costruire target e loss di training; la NMS confronta predizioni e score; il matching valutativo confronta predizioni e GT per contare TP, FP e FN secondo le regole fissate. Qui box e score sono assegnati e non si esegue un detector: i conti non misurano qualità, generalizzazione o velocità di un modello reale. C27 può conservare il registro per uso personale, senza alcun invio per la Lezione 15.

### Sintesi già svolta F01–F04

- **F01 — Coordinate:** il box (4, 6, 12, 10) diventa (11, 10, 27, 12) e l’inversa restituisce l’originale con errore 0, sotto le convenzioni di C05.
- **F02 — NMS:** con la stessa classe, soglia 0.5 conserva [0, 2]; soglia 0.9 conserva [0, 1, 2]. Se il secondo box ha classe diversa, la NMS class-aware conserva tutti a entrambe le soglie. Il confronto è già ragionato nella scheda di sintesi.
- **F03 — AP:** a IoU valutativa 0.5, prima della NMS AP integrale=5/6 e AP101=337/404; dopo NMS valgono entrambe 1, pur restando p5 FP. C17–C23 mostrano ranking e cumulati.
- **F04 — Errori:** a IoU valutativa 0.5, p2 è duplicato e p5 è sfondo; non ci sono FN. Nel caso distinto a IoU 0.85, i GT di B e C sono FN perché p3 e p4 non raggiungono la soglia.

Non occorre preparare una scheda aggiuntiva o consegnare il notebook e il JSON. In Colab si può scaricare il registro dal pannello File per conservarne una copia personale. D01–D08 della dispensa sono facoltativi per studio e ripasso; checkpoint e discussione conclusiva servono alla discussione e all’auto-verifica. Le istruzioni del lavoro individuale successivo sono nel notebook della Lezione 16.

In [ ]:
# C27
RESULTS['elapsed_python_seconds']=time.perf_counter()-START
out=Path('risultati_lezione15.json');out.write_text(json.dumps(RESULTS,indent=2,ensure_ascii=False),encoding='utf-8')
print('Creato',out.resolve());print('Tempo include lettura/interazione se eseguito a mano, non latenza detector.')
